In [1]:
!pip install -q transformers datasets sentencepiece accelerate
!pip install -q scikit-learn pandas tqdm

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig
)

import pandas as pd
import numpy as np
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [2]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [3]:
sample_requests = [
    "GET /login id NUM token TOKEN",
    "GET /product id NUM",
    "POST /checkout cart_id NUM",
    "GET /search query TEXT",
]

sample_requests

['GET /login id NUM token TOKEN',
 'GET /product id NUM',
 'POST /checkout cart_id NUM',
 'GET /search query TEXT']

In [4]:
enc = tokenizer(
    sample_requests,
    padding=True,
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

enc

{'input_ids': tensor([[  101,  2131,  1013,  8833,  2378,  8909, 16371,  2213, 19204, 19204,
           102],
        [  101,  2131,  1013,  4031,  8909, 16371,  2213,   102,     0,     0,
             0],
        [  101,  2695,  1013,  4638,  5833, 11122,  1035,  8909, 16371,  2213,
           102],
        [  101,  2131,  1013,  3945, 23032,  3793,   102,     0,     0,     0,
             0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]])}

In [5]:
class WAFDataset(Dataset):

    def __init__(self, texts, tokenizer, max_len=64):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = self.texts[idx]

        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze()
        }

In [6]:
dataset = WAFDataset(sample_requests, tokenizer)

loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True
)

batch = next(iter(loader))
batch

{'input_ids': tensor([[  101,  2695,  1013,  4638,  5833, 11122,  1035,  8909, 16371,  2213,
            102,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0],
         [  101,  2131,  1013,  3945, 23032,  3793,   102,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,   

In [7]:
class WAFTransformer(nn.Module):

    def __init__(self, model_name):

        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)

        hidden = self.encoder.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 1)
        )

    def forward(self, input_ids, attention_mask):

        out = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls = out.last_hidden_state[:,0]

        score = self.classifier(cls)

        return score

In [8]:
model = WAFTransformer(MODEL_NAME)
model = model.to(device)

model

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


WAFTransformer(
  (encoder): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (lin1): Line

In [9]:
batch = next(iter(loader))

input_ids = batch["input_ids"].to(device)
mask = batch["attention_mask"].to(device)

with torch.no_grad():
    score = model(input_ids, mask)

score

tensor([[-0.0387],
        [-0.0427],
        [-0.0427],
        [-0.0454]], device='cuda:0')

In [10]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-5
)

In [11]:
import re

def normalize_request(method, path, query):

    path = path.lower()

    # normalize numbers
    path = re.sub(r'\d+', 'NUM', path)

    tokens = [f"METHOD_{method}", f"PATH_{path}"]

    if query:
        params = query.split("&")

        for p in params:
            if "=" in p:
                k,v = p.split("=")

                if v.isdigit():
                    tokens.append(f"PARAM_{k}")
                    tokens.append("NUM")
                else:
                    tokens.append(f"PARAM_{k}")
                    tokens.append("TEXT")

    return " ".join(tokens)

In [12]:
def train_epoch(model, loader):

    model.train()

    total_loss = 0

    for batch in tqdm(loader):

        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)

        labels = torch.zeros(input_ids.size(0)).to(device)

        logits = model(input_ids, mask).squeeze()

        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
torch.save(model.state_dict(), "waf_transformer.pt")